In [1]:
from modules.ingest import load_faq_data, build_indices
from modules.rag_helper import RAGBase, VectorRAG, INSTRUCTIONS, PROMPT_TEMPLATE
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

# 1. Load data and build BOTH indices
documents = load_faq_data()
keyword_index, vector_index = build_indices(documents)

# 2. Initialize both RAG systems side-by-side
client = OpenAI()

keyword_rag = RAGBase(
    index=keyword_index,
    llm_client=client,
  
)

vector_rag = VectorRAG(
    index=vector_index,
    llm_client=client,
  
)

# ==========================================
# 3. THE TOGGLE 
# ==========================================
# (If you build a UI later, this maps perfectly to a dropdown menu)
SEARCH_MODE = "vector"  # Change to "keyword" to switch instantly!

# Route to the correct system
if SEARCH_MODE == "":
    active_rag = vector_rag
else:
    active_rag = keyword_rag

# ==========================================
# 4. Execute the Search
# ==========================================
# Because both classes use the exact same inputs, this code never changes!
answer = active_rag.rag(
    query="How much for a group lesson?", 
    filter_dict={'subcategory': 'group'}
)

print(f"[{SEARCH_MODE.upper()} SEARCH RESULTS]")
print(answer)

Building Keyword Index...
Building Vector Index...
Both indices built successfully!
[VECTOR SEARCH RESULTS]
Group lessons cost £12 per hour per student. For a full group of 5, the total session cost is £60 per hour. This setup is designed to be more affordable while focusing on core GCSE exam practice.


In [2]:
query='Does Wali know how to teach pythagoras theorem?'
active_rag.rag(query)

"I don't have that information. Please contact Wali directly via Phone or WhatsApp at 07737889846."

In [7]:
active_rag.rag('I just discovered the course, can I still join?')

"I don't have that information. Please contact Wali directly via Phone or WhatsApp at 07737889846."

In [8]:
active_rag.rag('how much do you charge per hour?')

"For an individual student, the charge for private 1-to-1 GCSE tutoring is £30 per hour. If you're interested in group sessions, the rate is £12 per hour per student."

In [9]:
active_rag.rag('Who is Wali?')

'Wali is an online GCSE and KS3 Maths tutor based in the UK. He supports the AQA, Edexcel, and OCR exam boards, focusing his 60-minute lessons on step-by-step explanations to help students regain confidence, refine exam techniques, and improve in specific topics.'

# Failed Questions
### when using just keyword search only

In [10]:
active_rag.rag('How many years experience have you been teaching in Mathematics and what level?')

"I don't have that information. Please contact Wali directly via Phone or WhatsApp at 07737889846."

In [11]:
active_rag.rag('How many years experience have you been teaching in Mathematics?')

"I don't have that information. Please contact Wali directly via Phone or WhatsApp at 07737889846."

In [12]:
active_rag.rag('Does Wali know how to teach pythagoras theorem?')

"I don't have that information. Please contact Wali directly via Phone or WhatsApp at 07737889846."

In [13]:
response = active_rag.rag(
    query="Tell me about your fees and costs.",
    filter_dict={'category': 'pricing'}
)
response

'The fees for private, 1-to-1 GCSE maths tutoring are £30 per hour. There are no hidden registration fees; the service is simply billed at this hourly rate.'

## checking categorise

In [14]:
import json
import pandas as pd

In [15]:
with open("knowledge-base.json", "r") as f:
    raw_data = json.load(f)

df = pd.DataFrame(raw_data)

# ==========================================
# Phase 1: Create the Knowledge Base
# ==========================================
# We only want unique context chunks to ingest into Elasticsearch.
# If we ingest duplicates, our vector search will return the same text multiple times.

# Drop duplicates based on the chunk_id
kb_df = df[['chunk_id', 'category', 'subcategory', 'mapped_context']].drop_duplicates(subset=['chunk_id'])

In [16]:
kb_df.category.unique()

<StringArray>
[          'tutor_profile',        'lesson_structure',
             'exam_boards',                 'pricing',
            'availability',           'lesson_format',
       'teaching_approach', 'objections_and_concerns',
       'business_policies',     'booking_and_contact']
Length: 10, dtype: str

In [17]:
active_rag.rag(
          query='How many years experience have you been teaching in Mathematics?',
              filter_dict={'category': 'tutor_profile'}
             )

"I don't have that information. Please contact Wali directly via Phone or WhatsApp at 07737889846."